# __Цифровые инструменты финансиста__

## __День 2.__ Работа с данными

### __2. Продвинутый анализ данных: загрузка__

#### 2.1. Внешние данные

In [1]:
import zipfile
from pathlib import Path
import pandas as pd
from urllib.request import urlretrieve
from IPython.display import display

def download_and_extract_data(url: str, output_dir: str = "data") -> list:
    """
    Загружает ZIP-файл по ссылке и распаковывает в указанную директорию.
    
    Args:
        url: URL ссылка на ZIP-файл
        output_dir: Директория для сохранения и распаковки файлов
    
    Returns:
        list: Список путей к извлеченным файлам
    """
    # Создаем директорию для данных
    data_dir = Path(output_dir)
    data_dir.mkdir(exist_ok=True)
    
    # Формируем путь для сохранения ZIP-файла
    zip_filename = data_dir / "downloaded_data.zip"
    
    # Загружаем файл
    print(f"Загрузка файла из {url}...")
    urlretrieve(url, zip_filename)
    print(f"Файл сохранен: {zip_filename}")
    
    # Распаковываем архив
    extracted_files = []
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        print(f"Распаковка архива в {data_dir}...")
        zip_ref.extractall(data_dir)
        extracted_files = [data_dir / name for name in zip_ref.namelist()]
        print(f"Распаковано файлов: {len(extracted_files)}")
    
    # Удаляем ZIP-файл после распаковки (опционально)
    # zip_filename.unlink()
    
    return extracted_files

def find_data_files(directory: str, extensions: list = None) -> pd.DataFrame:
    """
    Находит файлы данных в указанной директории.
    
    Args:
        directory: Директория для поиска
        extensions: Список расширений файлов для поиска (по умолчанию: .csv, .xlsx, .xls, .txt)
    
    Returns:
        pd.DataFrame: DataFrame с информацией о найденных файлах
    """
    if extensions is None:
        extensions = ['.csv', '.xlsx', '.xls', '.txt', '.json', '.parquet']
    
    data_dir = Path(directory)
    files_info = []
    
    for ext in extensions:
        for file_path in data_dir.rglob(f"*{ext}"):
            files_info.append({
                'filename': file_path.name,
                'path': str(file_path),
                'extension': ext,
                'size_mb': round(file_path.stat().st_size / (1024 * 1024), 2)
            })
    
    df_files = pd.DataFrame(files_info)
    return df_files

# Выполнение основной задачи
if __name__ == "__main__":
    # URL для загрузки
    url = "https://storage.yandexcloud.net/tochno-st-catalog/Rosstat/data_bdmo_118_v20250918/by_section/data_section35_112_v20250918_section_file.zip"
    
    # Загружаем и распаковываем архив
    extracted = download_and_extract_data(url)
    
    print("\n" + "="*60)
    print("ПОИСК ФАЙЛОВ ДАННЫХ")
    print("="*60)
    
    # Находим все файлы данных
    data_files = find_data_files("data")
    
    if len(data_files) > 0:
        print(f"\nНайдено файлов данных: {len(data_files)}")
        display(data_files)
        
        # Дополнительная информация о содержимом
        print("\n" + "-"*40)
        print("ПРЕДПРОСМОТР ПЕРВОГО ФАЙЛА")
        print("-"*40)
        
        first_file = data_files.iloc[0]
        print(f"\nФайл: {first_file['filename']}")
        print(f"Размер: {first_file['size_mb']} MB")
        
        # Пытаемся прочитать первый найденный файл
        try:
            if first_file['extension'] == '.csv':
                df_sample = pd.read_csv(first_file['path'], nrows=5, sep=";")
            elif first_file['extension'] in ['.xlsx', '.xls']:
                df_sample = pd.read_excel(first_file['path'], nrows=5)
            else:
                df_sample = None
            
            if df_sample is not None:
                print(f"\nПервые 5 строк:")
                display(df_sample)
        except Exception as e:
            print(f"Не удалось прочитать файл: {e}")
    else:
        print("Файлы данных не найдены в указанных расширениях")

Загрузка файла из https://storage.yandexcloud.net/tochno-st-catalog/Rosstat/data_bdmo_118_v20250918/by_section/data_section35_112_v20250918_section_file.zip...
Файл сохранен: data/downloaded_data.zip
Распаковка архива в data...
Распаковано файлов: 3

ПОИСК ФАЙЛОВ ДАННЫХ

Найдено файлов данных: 3


,filename,path,extension,size_mb
0,data_section35_112_v20250918.csv,data/data_section35_112_v20250918.csv,.csv,134.89
1,data_section35_112_v20250918.xlsx,data/data_section35_112_v20250918.xlsx,.xlsx,20.40
2,data_section35_112_v20250918.parquet,data/data_section35_112_v20250918.parquet,.parquet,1.26



----------------------------------------
ПРЕДПРОСМОТР ПЕРВОГО ФАЙЛА
----------------------------------------

Файл: data_section35_112_v20250918.csv
Размер: 134.89 MB

Первые 5 строк:


,indicator_section_code,indicator_section,indicator_code,indicator_name,zagr,region_id,region_name,mun_level,mun_district,municipality,...,mun_type_oktmo,oktmo_stable,oktmo_history,oktmo_year_from,oktmo_year_to,year,indicator_value,indicator_unit,indicator_period,comment
0,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Муниципальный округ город Славгород,Муниципальный округ город Славгород,...,Муниципальный округ,1539000,"Изменен тип и название, Объединение",2023,2025,2024,57594.0,Тысяча рублей,Значение показателя за год,CD
1,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,1601000,Без изменений,2010,2025,2008,0.0,Тысяча рублей,Значение показателя за год,CD
2,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,1601000,Без изменений,2010,2025,2009,0.0,Тысяча рублей,Значение показателя за год,CD
3,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,1601000,Без изменений,2010,2025,2023,1792.0,Тысяча рублей,Значение показателя за год,CD
4,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,1601000,Без изменений,2010,2025,2024,5674.0,Тысяча рублей,Значение показателя за год,CD


In [2]:
import pandas as pd
from IPython.display import display

def describe_data(file_path: str) -> None:
    """
    Выводит краткое описание данных из CSV-файла.
    
    Args:
        file_path: Путь к CSV-файлу с разделителем ';'
    """
    # Загрузка данных
    df = pd.read_csv(file_path, sep=';')
    
    # Базовая информация
    print("=" * 79)
    print("ОПИСАНИЕ ДАННЫХ")
    print("=" * 79)
    
    # Размерность
    print(f"\n1. РАЗМЕРНОСТЬ:")
    print(f"   - Количество строк: {df.shape[0]:,}")
    print(f"   - Количество столбцов: {df.shape[1]}")
    
    # Структура
    print(f"\n2. СТРУКТУРА ДАННЫХ:")
    print(f"   - Индекс: {df.index.name if df.index.name else 'RangeIndex'}")
    print(f"   - Типы данных:\n{df.dtypes.value_counts()}")
    
    # Колонки
    print(f"\n3. КОЛОНКИ ({len(df.columns)} шт.):")
    display(df.head(3))
    
    # Статистика
    print(f"\n4. СТАТИСТИКА ПО ЧИСЛОВЫМ КОЛОНКАМ:")
    display(df.describe())
    
    # Пропуски
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_report = pd.DataFrame({
        'Пропуски': missing,
        'Доля (%)': missing_pct
    }).sort_values('Пропуски', ascending=False)
    missing_report = missing_report[missing_report['Пропуски'] > 0]
    
    if not missing_report.empty:
        print(f"\n5. ПРОПУСКИ:")
        display(missing_report)
    else:
        print(f"\n5. ПРОПУСКИ: отсутствуют")
    
    # Дубликаты
    duplicates = df.duplicated().sum()
    print(f"\n6. ДУБЛИКАТЫ: {duplicates:,} шт.")
    
    # Память
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"\n7. ИСПОЛЬЗОВАНИЕ ПАМЯТИ: {memory_mb:.2f} MB")
    
    print("\n" + "=" * 79)

# Выполнение
describe_data('./data/data_section35_112_v20250918.csv')

ОПИСАНИЕ ДАННЫХ

1. РАЗМЕРНОСТЬ:
   - Количество строк: 178,475
   - Количество столбцов: 22

2. СТРУКТУРА ДАННЫХ:
   - Индекс: RangeIndex
   - Типы данных:
object     17
int64       4
float64     1
Name: count, dtype: int64

3. КОЛОНКИ (22 шт.):


,indicator_section_code,indicator_section,indicator_code,indicator_name,zagr,region_id,region_name,mun_level,mun_district,municipality,...,mun_type_oktmo,oktmo_stable,oktmo_history,oktmo_year_from,oktmo_year_to,year,indicator_value,indicator_unit,indicator_period,comment
0,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Муниципальный округ город Славгород,Муниципальный округ город Славгород,...,Муниципальный округ,01539000,"Изменен тип и название, Объединение",2023,2025,2024,57594.0,Тысяча рублей,Значение показателя за год,CD
1,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,01601000,Без изменений,2010,2025,2008,0.0,Тысяча рублей,Значение показателя за год,CD
2,35,Охрана окружающей среды,Y48055001,Текущие (эксплуатационные) затраты на охрану о...,CD,1,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,01601000,Без изменений,2010,2025,2009,0.0,Тысяча рублей,Значение показателя за год,CD



4. СТАТИСТИКА ПО ЧИСЛОВЫМ КОЛОНКАМ:


,indicator_section_code,region_id,oktmo,year,indicator_value
count,178475.0,178475.000000,1.784750e+05,178475.000000,1.783920e+05
mean,35.0,47.513825,4.815995e+07,2014.334445,4.780231e+04
std,0.0,28.098455,2.810169e+07,3.400568,5.078278e+05
min,35.0,1.000000,1.539000e+06,2008.000000,0.000000e+00
25%,35.0,22.000000,2.275800e+07,2012.000000,1.680000e-01
50%,35.0,46.000000,4.676200e+07,2015.000000,3.000000e+00
75%,35.0,71.000000,7.161800e+07,2016.000000,7.000000e+01
max,35.0,99.000000,9.970100e+07,2024.000000,7.307020e+07



5. ПРОПУСКИ:


,Пропуски,Доля (%)
indicator_value,83,0.046505



6. ДУБЛИКАТЫ: 0 шт.

7. ИСПОЛЬЗОВАНИЕ ПАМЯТИ: 440.96 MB



#### 2.2. Попробуем в большие данные

Будем загружать много файлов файлов из нашего источника и формировать из них единый датасет. Наш источник - [Московская межбанковская валютная биржа](https://www.moex.com/ru/marketdata/archive/#/engine=stock&market=index&data_type=main&data_interval=current&data_format=csv&year=2026) (итоги дневных сессий).

Если посмотреть на ссылки на нужные нам файлы, то можно заметить определенные закономерности (возьмем первые три ссылки):
- https://iss.moex.com/iss/downloads/engines/stock/markets/index/sessions/main/years/2026/months/04/days/07/securities_moex_stock_index_main_2026_04_07.csv.zip
- https://iss.moex.com/iss/downloads/engines/stock/markets/index/sessions/main/years/2026/months/04/days/06/securities_moex_stock_index_main_2026_04_06.csv.zip
- https://iss.moex.com/iss/downloads/engines/stock/markets/index/sessions/main/years/2026/months/04/days/05/securities_moex_stock_index_main_2026_04_05.csv.zip

In [3]:
"""
Скрипт для загрузки файлов MOEX за февраль, март, апрель 2026 года.
Загружает данные за все дни указанных месяцев.
"""

import os
import urllib.request
import urllib.error
from datetime import datetime, timedelta
from typing import List, Tuple


def generate_urls(year: int, month: int) -> List[Tuple[str, str]]:
    """
    Генерирует список URL и имен файлов для заданного месяца.
    
    Args:
        year: Год (например, 2026)
        month: Месяц (1-12)
    
    Returns:
        Список кортежей (url, имя_файла)
    """
    urls = []
    
    # Определяем количество дней в месяце
    if month == 12:
        next_month = datetime(year + 1, 1, 1)
    else:
        next_month = datetime(year, month + 1, 1)
    
    first_day = datetime(year, month, 1)
    last_day = next_month - timedelta(days=1)
    
    # Генерируем URL для каждого дня месяца
    current_date = first_day
    while current_date <= last_day:
        day = current_date.day
        month_str = f"{month:02d}"
        day_str = f"{day:02d}"
        
        url = (f"https://iss.moex.com/iss/downloads/engines/stock/markets/index/"
               f"sessions/main/years/{year}/months/{month_str}/days/{day_str}/"
               f"securities_moex_stock_index_main_{year}_{month_str}_{day_str}.csv.zip")
        
        filename = f"securities_moex_stock_index_main_{year}_{month_str}_{day_str}.csv.zip"
        urls.append((url, filename))
        
        current_date += timedelta(days=1)
    
    return urls


def download_file(url: str, filename: str, folder: str = "bigdata") -> bool:
    """
    Скачивает файл по URL и сохраняет в указанную папку.
    
    Args:
        url: URL для скачивания
        filename: Имя сохраняемого файла
        folder: Папка для сохранения (по умолчанию 'bigdata')
    
    Returns:
        True если файл успешно скачан, False если файл не найден
    """
    # Создаем папку если она не существует
    os.makedirs(folder, exist_ok=True)
    
    filepath = os.path.join(folder, filename)
    
    try:
        print(f"Попытка загрузить: {filename}")
        urllib.request.urlretrieve(url, filepath)
        print(f"✓ Успешно загружен: {filename}")
        return True
    except urllib.error.HTTPError as e:
        if e.code == 404:
            print(f"✗ Файл не найден: {filename}")
        else:
            print(f"✗ Ошибка {e.code} при загрузке {filename}")
        return False
    except Exception as e:
        print(f"✗ Ошибка при загрузке {filename}: {str(e)}")
        return False


def download_month_data(year: int, month: int) -> dict:
    """
    Загружает все доступные файлы за указанный месяц.
    
    Args:
        year: Год
        month: Месяц
    
    Returns:
        Словарь со статистикой загрузки
    """
    month_name = datetime(year, month, 1).strftime("%B")
    print(f"\n{'='*60}")
    print(f"Загрузка данных за {month_name} {year}")
    print(f"{'='*60}")
    
    urls = generate_urls(year, month)
    
    stats = {
        "month": month,
        "year": year,
        "total": len(urls),
        "downloaded": 0,
        "failed": 0
    }
    
    for url, filename in urls:
        if download_file(url, filename):
            stats["downloaded"] += 1
        else:
            stats["failed"] += 1
    
    print(f"\nИтоги за {month_name}:")
    print(f"  Всего дней: {stats['total']}")
    print(f"  Загружено: {stats['downloaded']}")
    print(f"  Пропущено: {stats['failed']}")
    
    return stats


def main():
    """Основная функция для загрузки данных за февраль, март, апрель 2026."""
    print("Начало загрузки данных MOEX")
    print("="*60)
    
    year = 2026
    months = [2, 3, 4]  # Февраль, Март, Апрель
    all_stats = []
    
    for month in months:
        stats = download_month_data(year, month)
        all_stats.append(stats)
    
    # Итоговая статистика
    print(f"\n{'='*60}")
    print("ОБЩИЕ ИТОГИ ЗАГРУЗКИ")
    print(f"{'='*60}")
    
    total_files = sum(s["total"] for s in all_stats)
    total_downloaded = sum(s["downloaded"] for s in all_stats)
    
    print(f"Всего проверено файлов: {total_files}")
    print(f"Успешно загружено: {total_downloaded}")
    print(f"Пропущено (нет данных): {total_files - total_downloaded}")
    
    # Проверяем содержимое папки
    if os.path.exists("bigdata"):
        files = os.listdir("bigdata")
        zip_files = [f for f in files if f.endswith('.zip')]
        print(f"\nФайлы сохранены в папке 'bigdata':")
        print(f"Всего ZIP-файлов: {len(zip_files)}")
        for f in sorted(zip_files)[:5]:  # Показываем первые 5
            print(f"  - {f}")
        if len(zip_files) > 5:
            print(f"  ... и еще {len(zip_files) - 5} файлов")


# Запуск загрузки
if __name__ == "__main__":
    main()

Начало загрузки данных MOEX

Загрузка данных за February 2026
Попытка загрузить: securities_moex_stock_index_main_2026_02_01.csv.zip
✗ Файл не найден: securities_moex_stock_index_main_2026_02_01.csv.zip
Попытка загрузить: securities_moex_stock_index_main_2026_02_02.csv.zip
✗ Файл не найден: securities_moex_stock_index_main_2026_02_02.csv.zip
Попытка загрузить: securities_moex_stock_index_main_2026_02_03.csv.zip
✗ Файл не найден: securities_moex_stock_index_main_2026_02_03.csv.zip
Попытка загрузить: securities_moex_stock_index_main_2026_02_04.csv.zip
✗ Файл не найден: securities_moex_stock_index_main_2026_02_04.csv.zip
Попытка загрузить: securities_moex_stock_index_main_2026_02_05.csv.zip
✗ Файл не найден: securities_moex_stock_index_main_2026_02_05.csv.zip
Попытка загрузить: securities_moex_stock_index_main_2026_02_06.csv.zip
✗ Файл не найден: securities_moex_stock_index_main_2026_02_06.csv.zip
Попытка загрузить: securities_moex_stock_index_main_2026_02_07.csv.zip
✗ Файл не найден: sec

In [5]:
"""Обработка ZIP-архивов с данными MOEX и объединение в единый CSV файл."""

import zipfile
import pandas as pd
from pathlib import Path
from datetime import datetime


def process_zip_files(bigdata_dir='bigdata'):
    """Извлекает данные из всех ZIP-файлов и объединяет в один CSV файл.
    
    Args:
        bigdata_dir: Путь к директории с ZIP-файлами.
    
    Returns:
        tuple: (путь к сохраненному файлу, дата начала, дата окончания)
    """
    # Список для хранения DataFrame из каждого файла
    all_dfs = []
    all_dates = []
    
    # Получаем все ZIP-файлы в директории
    zip_path = Path(bigdata_dir)
    zip_files = list(zip_path.glob('*.zip'))
    
    if not zip_files:
        raise FileNotFoundError(f'Нет ZIP-файлов в директории {bigdata_dir}')
    
    print(f'Найдено ZIP-файлов: {len(zip_files)}')
    
    # Обрабатываем каждый ZIP-файл
    for i, zip_file in enumerate(zip_files, 1):
        print(f'\nОбработка файла {i}/{len(zip_files)}: {zip_file.name}')
        
        # Открываем ZIP-архив
        with zipfile.ZipFile(zip_file, 'r') as zf:
            # Получаем список CSV-файлов в архиве
            csv_files = [f for f in zf.namelist() if f.endswith('.csv')]
            
            if not csv_files:
                print(f'  В архиве {zip_file.name} нет CSV-файлов')
                continue
            
            # Обрабатываем каждый CSV в архиве
            for csv_file in csv_files:
                print(f'  Извлечение: {csv_file}')
                
                # Извлекаем данные из CSV
                with zf.open(csv_file) as f:
                    df = pd.read_csv(
                        f,
                        encoding='cp1251',
                        sep=';',
                        header=1,
                    )
                
                # Преобразуем TRADEDATE в datetime для определения границ
                if 'TRADEDATE' in df.columns:
                    df['TRADEDATE'] = pd.to_datetime(
                        df['TRADEDATE'], format='%d.%m.%Y'
                    )
                    # Сохраняем даты для определения START и END
                    all_dates.extend(df['TRADEDATE'].unique())
                
                # Добавляем DataFrame в список
                all_dfs.append(df)
                
                print(f'    Добавлено строк: {len(df)}')
    
    # Проверяем, есть ли данные
    if not all_dfs:
        raise ValueError('Не удалось извлечь данные из ZIP-архивов')
    
    # Объединяем все DataFrame
    print('\nОбъединение данных...')
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    # Удаляем дубликаты, если они есть
    initial_count = len(combined_df)
    combined_df = combined_df.drop_duplicates()
    print(f'Удалено дубликатов: {initial_count - len(combined_df)}')
    
    # Определяем START и END
    if all_dates:
        start_date = min(all_dates).strftime('%Y%m%d')
        end_date = max(all_dates).strftime('%Y%m%d')
    else:
        # Альтернативный способ через преобразование TRADEDATE
        if 'TRADEDATE' in combined_df.columns:
            start_date = combined_df['TRADEDATE'].min().strftime('%Y%m%d')
            end_date = combined_df['TRADEDATE'].max().strftime('%Y%m%d')
        else:
            raise ValueError('Нет колонки с датами TRADEDATE')
    
    # Сохраняем объединенный файл
    output_filename = f'moex_data_{start_date}_{end_date}.csv'
    combined_df.to_csv(output_filename, index=False, encoding='utf-8')
    
    print(f'\nРезультаты:')
    print(f'  Всего строк: {len(combined_df)}')
    print(f'  Колонок: {len(combined_df.columns)}')
    print(f'  Период: {start_date} - {end_date}')
    print(f'  Сохранено в: {output_filename}')
    
    return output_filename, start_date, end_date


def display_data_preview(file_path, n_rows=5):
    """Отображает первые строки сохраненного файла.
    
    Args:
        file_path: Путь к CSV файлу.
        n_rows: Количество строк для отображения.
    """
    if not Path(file_path).exists():
        print(f'Файл {file_path} не найден')
        return
    
    df_preview = pd.read_csv(file_path, nrows=n_rows)
    
    print(f'\nПредпросмотр данных ({n_rows} первых строк):')
    display(df_preview)  # Используем display для Jupyter


# Основной блок выполнения
if __name__ == '__main__' or 'get_ipython' in globals():
    try:
        # Обрабатываем ZIP-файлы
        output_file, start, end = process_zip_files()
        
        # Показываем предпросмотр результата
        display_data_preview(output_file)
        
        # Дополнительная информация о данных
        df_info = pd.read_csv(output_file)
        print(f'\nСтатистика по данным:')
        print(f'  Уникальных SECID: {df_info["SECID"].nunique()}')
        print(f'  Диапазон дат: {df_info["TRADEDATE"].min()} - {df_info["TRADEDATE"].max()}')
        
    except Exception as e:
        print(f'Ошибка: {e}')

Найдено ZIP-файлов: 30

Обработка файла 1/30: securities_moex_stock_index_main_2026_03_24.csv.zip
  Извлечение: securities.csv
    Добавлено строк: 520

Обработка файла 2/30: securities_moex_stock_index_main_2026_04_14.csv.zip
  Извлечение: securities.csv
    Добавлено строк: 520

Обработка файла 3/30: securities_moex_stock_index_main_2026_03_16.csv.zip
  Извлечение: securities.csv
    Добавлено строк: 536

Обработка файла 4/30: securities_moex_stock_index_main_2026_03_15.csv.zip
  Извлечение: securities.csv
    Добавлено строк: 1

Обработка файла 5/30: securities_moex_stock_index_main_2026_04_02.csv.zip
  Извлечение: securities.csv
    Добавлено строк: 533

Обработка файла 6/30: securities_moex_stock_index_main_2026_03_29.csv.zip
  Извлечение: securities.csv
    Добавлено строк: 1

Обработка файла 7/30: securities_moex_stock_index_main_2026_03_17.csv.zip
  Извлечение: securities.csv
    Добавлено строк: 527

Обработка файла 8/30: securities_moex_stock_index_main_2026_03_20.csv.zip
  И

,BOARDID,SECID,TRADEDATE,SHORTNAME,NAME,CLOSE,OPEN,HIGH,LOW,VALUE,DURATION,YIELD,DECIMALS,CAPITALIZATION,CURRENCYID,DIVISOR,TRADINGSESSION,VOLUME,TRADE_SESSION_DATE,RECALC_DATE
0,INAV,2xEQT,2026-03-24,iNAV 2xEQT,Расчетная цена одного пая интервального ПИФа «...,"917,11","917,11","917,11","917,11",0,0.0,0,2,"1488303778,245",RUB,NaN,1,"0,0000",24.03.2026,NaN
1,INAV,2xOFZ,2026-03-24,iNAV 2xOFZ,Расчетная цена одного пая интервального ПИФа «...,"156721,19","157153,88","157750,03","156680,42",0,0.0,0,2,"3046880947,96",RUB,NaN,1,"0,0000",24.03.2026,NaN
2,INAV,AKAIA,2026-03-24,iNAV Альфа Антиинфляционный,"Расчётная цена одного пая Биржевого ПИФа РФИ ""...","107,3876","107,1147","107,8408","107,0571",0,0.0,0,4,"21426488,296",RUB,NaN,1,"0,0000",24.03.2026,NaN
3,INAV,AKBCA,2026-03-24,iNAV Альфа-Капитал Голубые фишки,Расчетная цена одного пая Биржевого ПИФа РФИ «...,"104,947","104,6319","105,2215","104,3547",0,0.0,0,4,"112756196,14",RUB,NaN,1,"0,0000",24.03.2026,NaN
4,INAV,AKCNA,2026-03-24,iNAV Альфа Потреб. сектор,Расчетная цена одного пая Биржевого ПИФа РФИ «...,"96,3369","96,3369","96,3369","96,3369",0,0.0,0,4,"24565904,73",RUB,NaN,1,"0,0000",24.03.2026,NaN



Статистика по данным:
  Уникальных SECID: 557
  Диапазон дат: 2026-03-12 - 2026-04-15


In [8]:
import pandas as pd
from IPython.display import display

# Загрузка данных
df = pd.read_csv('moex_data_20260312_20260415.csv')

# Краткое описание данных
print("=" * 60)
print("ОПИСАНИЕ ДАННЫХ")
print("=" * 60)

# 1. Основная информация
print("\n1. РАЗМЕР ДАННЫХ:")
print(f"   - Количество строк: {df.shape[0]:,}")
print(f"   - Количество столбцов: {df.shape[1]}")

# 2. Информация о столбцах
print("\n2. ИНФОРМАЦИЯ О СТОЛБЦАХ:")
df_info = pd.DataFrame({
    'Тип данных': df.dtypes,
    'Непустых значений': df.count(),
    'Пустых значений': df.isnull().sum(),
    'Уникальных значений': df.nunique()
})
display(df_info)

# 3. Статистика для числовых столбцов
numeric_cols = df.select_dtypes(include=['number']).columns
if len(numeric_cols) > 0:
    print("\n3. СТАТИСТИКА ЧИСЛОВЫХ ДАННЫХ:")
    display(df[numeric_cols].describe())
else:
    print("\n3. Числовые столбцы отсутствуют")

# 4. Первые 5 строк
print("\n4. ПЕРВЫЕ 5 СТРОК ДАННЫХ:")
display(df.head())

# 5. Последние 5 строк
print("\n5. ПОСЛЕДНИЕ 5 СТРОК ДАННЫХ:")
display(df.tail())

# 6. Проверка на дубликаты
duplicates = df.duplicated().sum()
print(f"\n6. КОЛИЧЕСТВО ДУБЛИКАТОВ: {duplicates}")

# 7. Объем памяти
memory_usage = df.memory_usage(deep=True).sum()
print(f"7. ОБЪЕМ ПАМЯТИ: {memory_usage / 1024**2:.2f} MB")

print("\n" + "=" * 60)
print("ОПИСАНИЕ ЗАВЕРШЕНО")
print("=" * 60)

ОПИСАНИЕ ДАННЫХ

1. РАЗМЕР ДАННЫХ:
   - Количество строк: 13,275
   - Количество столбцов: 20

2. ИНФОРМАЦИЯ О СТОЛБЦАХ:


,Тип данных,Непустых значений,Пустых значений,Уникальных значений
BOARDID,object,13275,0,6
SECID,object,13275,0,557
TRADEDATE,object,13275,0,30
SHORTNAME,object,13275,0,557
NAME,object,13225,50,552
CLOSE,object,13173,102,11297
OPEN,object,13173,102,11336
HIGH,object,13173,102,11314
LOW,object,13173,102,11307
VALUE,object,13083,192,5372



3. СТАТИСТИКА ЧИСЛОВЫХ ДАННЫХ:


,DURATION,DECIMALS,TRADINGSESSION,RECALC_DATE
count,12537.000000,13275.000000,13275.0,0.0
mean,277.312515,2.257100,1.0,NaN
std,538.318422,0.837007,0.0,NaN
min,0.000000,0.000000,1.0,NaN
25%,0.000000,2.000000,1.0,NaN
50%,0.000000,2.000000,1.0,NaN
75%,519.000000,2.000000,1.0,NaN
max,3028.000000,6.000000,1.0,NaN



4. ПЕРВЫЕ 5 СТРОК ДАННЫХ:


,BOARDID,SECID,TRADEDATE,SHORTNAME,NAME,CLOSE,OPEN,HIGH,LOW,VALUE,DURATION,YIELD,DECIMALS,CAPITALIZATION,CURRENCYID,DIVISOR,TRADINGSESSION,VOLUME,TRADE_SESSION_DATE,RECALC_DATE
0,INAV,2xEQT,2026-03-24,iNAV 2xEQT,Расчетная цена одного пая интервального ПИФа «...,"917,11","917,11","917,11","917,11",0,0.0,0,2,"1488303778,245",RUB,NaN,1,"0,0000",24.03.2026,NaN
1,INAV,2xOFZ,2026-03-24,iNAV 2xOFZ,Расчетная цена одного пая интервального ПИФа «...,"156721,19","157153,88","157750,03","156680,42",0,0.0,0,2,"3046880947,96",RUB,NaN,1,"0,0000",24.03.2026,NaN
2,INAV,AKAIA,2026-03-24,iNAV Альфа Антиинфляционный,"Расчётная цена одного пая Биржевого ПИФа РФИ ""...","107,3876","107,1147","107,8408","107,0571",0,0.0,0,4,"21426488,296",RUB,NaN,1,"0,0000",24.03.2026,NaN
3,INAV,AKBCA,2026-03-24,iNAV Альфа-Капитал Голубые фишки,Расчетная цена одного пая Биржевого ПИФа РФИ «...,"104,947","104,6319","105,2215","104,3547",0,0.0,0,4,"112756196,14",RUB,NaN,1,"0,0000",24.03.2026,NaN
4,INAV,AKCNA,2026-03-24,iNAV Альфа Потреб. сектор,Расчетная цена одного пая Биржевого ПИФа РФИ «...,"96,3369","96,3369","96,3369","96,3369",0,0.0,0,4,"24565904,73",RUB,NaN,1,"0,0000",24.03.2026,NaN



5. ПОСЛЕДНИЕ 5 СТРОК ДАННЫХ:


,BOARDID,SECID,TRADEDATE,SHORTNAME,NAME,CLOSE,OPEN,HIGH,LOW,VALUE,DURATION,YIELD,DECIMALS,CAPITALIZATION,CURRENCYID,DIVISOR,TRADINGSESSION,VOLUME,TRADE_SESSION_DATE,RECALC_DATE
13270,AGRO,WHCPT,2026-04-10,Индекс пшеницы НТБ,Индекс цен пшеницы по итогам товарных аукционо...,16668,16668,16668,16668,NaN,NaN,NaN,2,NaN,RUB,NaN,1,"1385,0000",10.04.2026,NaN
13271,RTSI,WHFOB,2026-04-10,WHFOB,Внебиржевой индекс пшеницы FOB глубоководные п...,"233,9","233,9","233,9","233,9",308548255,0.0,0,1,0,USD,NaN,1,"1319000,0000",10.04.2026,NaN
13272,INAV,WILDA,2026-04-10,iNAV ДОХОДЪ Анализ акций,"Расчетная цена одного пая Биржевого ПИФа РФИ ""...","1034,41","1034,41","1034,41","1034,41",0,0.0,0,2,"26377521,18",RUB,NaN,1,"0,0000",10.04.2026,NaN
13273,INAV,XUSDA,2026-04-10,iNAV XUSDA,Расчетная цена одного пая интервального ПИФа «...,"9723,87","9777,77","9777,77","9723,87",0,0.0,0,2,"528177332,636005",RUB,NaN,1,"0,0000",10.04.2026,NaN
13274,INAV,YUANB,2026-04-10,iNAV Облигации в юанях RUB,Расчётная цена одного пая Биржевого ПИФа РФИ «...,"116,51","116,91","116,91","116,24",0,0.0,0,2,"455516606,0407",RUB,NaN,1,"0,0000",10.04.2026,NaN



6. КОЛИЧЕСТВО ДУБЛИКАТОВ: 0
7. ОБЪЕМ ПАМЯТИ: 13.72 MB

ОПИСАНИЕ ЗАВЕРШЕНО


#### 2.3. Трансформация данных с Pandas и SQL

Оценим наши "большие" данные:

```prompt
## Базовые параметры
- роль: финансовый аналитик на Python
- уровень: начинающий
- температура: 0 (максимальная точность и предсказуемость)

## Входные данные
- датафрейм pandas с именем `df`
- установленный sqlite в окружении

## Задача
- загрузить датафрейм `df` в базу данных sqlite

## Технические ограничения
- код запускается в интерактивном ноутбуке Jupyter

## Технические требования
- архитектура: максимально упрощенная
- вывод данных: адаптированный под интерактивные ноутбуки, где возможно используй display вместо print 
- стиль кода: PEP 8, black (длина строки 79)
- документация: Docstrings в стиле Google
```

In [9]:
import sqlite3
import pandas as pd
from IPython.display import display

def load_df_to_sqlite(df: pd.DataFrame, db_name: str = 'database.db', 
                       table_name: str = 'data_table', if_exists: str = 'replace'):
    """Загружает DataFrame в SQLite базу данных.
    
    Args:
        df: DataFrame для загрузки
        db_name: Имя файла базы данных SQLite
        table_name: Имя таблицы в базе данных
        if_exists: Действие если таблица существует 
                   ('replace', 'append', 'fail')
    """
    # Создаем соединение с базой данных
    conn = sqlite3.connect(db_name)
    
    try:
        # Загружаем DataFrame в SQLite
        df.to_sql(table_name, conn, if_exists=if_exists, index=False)
        
        # Выводим информацию о загрузке
        display(f"✅ Данные успешно загружены в базу '{db_name}'")
        display(f"📊 Таблица: '{table_name}'")
        display(f"📏 Размер данных: {df.shape[0]} строк × {df.shape[1]} колонок")
        
        # Проверяем загруженные данные
        check_query = f"SELECT COUNT(*) as row_count FROM {table_name}"
        count_df = pd.read_sql_query(check_query, conn)
        display(f"🔍 Проверка: {count_df.iloc[0, 0]} строк в таблице")
        
    except Exception as e:
        display(f"❌ Ошибка при загрузке данных: {e}")
    finally:
        # Закрываем соединение
        conn.close()

# Загрузка DataFrame в базу данных
load_df_to_sqlite(df, 'my_database.db', 'my_table', 'replace')

"✅ Данные успешно загружены в базу 'my_database.db'"

"📊 Таблица: 'my_table'"

'📏 Размер данных: 13275 строк × 20 колонок'

'🔍 Проверка: 13275 строк в таблице'

In [11]:
import sqlite3
import pandas as pd
from IPython.display import display

def query_sqlite(db_name: str = 'my_database.db', 
                 query: str = None, 
                 table_name: str = None,
                 limit: int = 5):
    """Выполняет SQL запрос к SQLite базе данных.
    
    Args:
        db_name: Имя файла базы данных SQLite
        query: SQL запрос (если None, то формируется автоматически)
        table_name: Имя таблицы для простого SELECT запроса
        limit: Лимит строк для простого запроса
    
    Returns:
        DataFrame с результатами запроса
    """
    # Создаем соединение
    conn = sqlite3.connect(db_name)
    
    try:
        # Формируем запрос если не указан
        if query is None and table_name is not None:
            query = f"SELECT * FROM {table_name} LIMIT {limit}"
        elif query is None:
            raise ValueError("Укажите query или table_name")
        
        # Выполняем запрос
        display(f"📝 Выполняется запрос:\n{query}")
        result_df = pd.read_sql_query(query, conn)
        
        # Выводим результат
        display(f"📊 Результат: {len(result_df)} строк")
        display(result_df)
        
        return result_df
        
    except Exception as e:
        display(f"❌ Ошибка запроса: {e}")
        return None
    finally:
        conn.close()

# Примеры запросов

# Простой SELECT (первые 5 строк)
print("=" * 50)
print("Пример 1: Первые 5 строк")
print("=" * 50)
df_result = query_sqlite('my_database.db', table_name='my_table', limit=5)

Пример 1: Первые 5 строк


'📝 Выполняется запрос:\nSELECT * FROM my_table LIMIT 5'

'📊 Результат: 5 строк'

,BOARDID,SECID,TRADEDATE,SHORTNAME,NAME,CLOSE,OPEN,HIGH,LOW,VALUE,DURATION,YIELD,DECIMALS,CAPITALIZATION,CURRENCYID,DIVISOR,TRADINGSESSION,VOLUME,TRADE_SESSION_DATE,RECALC_DATE
0,INAV,2xEQT,2026-03-24,iNAV 2xEQT,Расчетная цена одного пая интервального ПИФа «...,"917,11","917,11","917,11","917,11",0,0.0,0,2,"1488303778,245",RUB,None,1,"0,0000",24.03.2026,None
1,INAV,2xOFZ,2026-03-24,iNAV 2xOFZ,Расчетная цена одного пая интервального ПИФа «...,"156721,19","157153,88","157750,03","156680,42",0,0.0,0,2,"3046880947,96",RUB,None,1,"0,0000",24.03.2026,None
2,INAV,AKAIA,2026-03-24,iNAV Альфа Антиинфляционный,"Расчётная цена одного пая Биржевого ПИФа РФИ ""...","107,3876","107,1147","107,8408","107,0571",0,0.0,0,4,"21426488,296",RUB,None,1,"0,0000",24.03.2026,None
3,INAV,AKBCA,2026-03-24,iNAV Альфа-Капитал Голубые фишки,Расчетная цена одного пая Биржевого ПИФа РФИ «...,"104,947","104,6319","105,2215","104,3547",0,0.0,0,4,"112756196,14",RUB,None,1,"0,0000",24.03.2026,None
4,INAV,AKCNA,2026-03-24,iNAV Альфа Потреб. сектор,Расчетная цена одного пая Биржевого ПИФа РФИ «...,"96,3369","96,3369","96,3369","96,3369",0,0.0,0,4,"24565904,73",RUB,None,1,"0,0000",24.03.2026,None


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13275 entries, 0 to 13274
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   BOARDID             13275 non-null  object 
 1   SECID               13275 non-null  object 
 2   TRADEDATE           13275 non-null  object 
 3   SHORTNAME           13275 non-null  object 
 4   NAME                13225 non-null  object 
 5   CLOSE               13173 non-null  object 
 6   OPEN                13173 non-null  object 
 7   HIGH                13173 non-null  object 
 8   LOW                 13173 non-null  object 
 9   VALUE               13083 non-null  object 
 10  DURATION            12537 non-null  float64
 11  YIELD               12537 non-null  object 
 12  DECIMALS            13275 non-null  int64  
 13  CAPITALIZATION      12562 non-null  object 
 14  CURRENCYID          13275 non-null  object 
 15  DIVISOR             5327 non-null   object 
 16  TRAD

Часто при работе над одним проектом нам приходится переключаться между SQL и Python. При этом некоторые из нас знакомы с управлением данными в SQL-запросах, но не на Python, что мешает нашей эффективности и производительности. На самом деле, используя Pandas, можно добиться на Python такого же результата и делать те же трансформации данных, как в SQL-запросах. И наоборот.

```prompt
Задача: отвечать на вопрос и предлагать трансформацию данных через операции с библиотекой pandas и одновременно аналогичные операции через запросы SQL

Вопрос: сгруппируй объемы торгов VALUE по ценным бумагам SECID

Данные: 
- датафрейм `df`
- таблица `my_table` в sqlite с полями как в датафрейме

Вывод df.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15630 entries, 0 to 15629
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   BOARDID             15630 non-null  object 
 1   SECID               15630 non-null  object 
 2   TRADEDATE           15630 non-null  object 
 3   SHORTNAME           15630 non-null  object 
 4   NAME                15570 non-null  object 
 5   CLOSE               15510 non-null  object 
 6   OPEN                15510 non-null  object 
 7   HIGH                15510 non-null  object 
 8   LOW                 15510 non-null  object 
 9   VALUE               15420 non-null  object 
 10  DURATION            14760 non-null  float64
 11  YIELD               14760 non-null  object 
 12  DECIMALS            15630 non-null  int64  
 13  CAPITALIZATION      14790 non-null  object 
 14  CURRENCYID          15630 non-null  object 
 15  DIVISOR             6150 non-null   object 
 16  TRADINGSESSION      15630 non-null  int64  
 17  VOLUME              14220 non-null  object 
 18  TRADE_SESSION_DATE  15630 non-null  object 
 19  RECALC_DATE         0 non-null      float64
 20  TRADEDATE_dt        15630 non-null  object 
dtypes: float64(2), int64(2), object(17)
memory usage: 2.5+ MB
```

In [13]:
# Преобразуем VALUE в числовой тип (обрабатывая возможные ошибки)
df['VALUE'] = pd.to_numeric(df['VALUE'], errors='coerce')

# Группировка и суммирование
result_pd = df.groupby('SECID')['VALUE'].sum().reset_index()

# Сортируем по убыванию VALUE для наглядности
result_pd = result_pd.sort_values('VALUE', ascending=False)

print(result_pd)

       SECID         VALUE
482  RUSFARN  3.267290e+13
462   RUSFAR  1.449858e+13
448    RUPAI  3.117474e+12
450    RUPMI  3.117474e+12
159   MCFWTR  1.776192e+12
..       ...           ...
274    RTSOG  0.000000e+00
82   FIXENPG  0.000000e+00
78   FIXCBOM  0.000000e+00
77   FIXBSPB  0.000000e+00
354   RUEUTR  0.000000e+00

[557 rows x 2 columns]


In [14]:
query = """
SELECT 
    SECID,
    SUM(CAST(VALUE AS FLOAT)) AS TOTAL_VALUE
FROM my_table
WHERE VALUE IS NOT NULL 
    AND VALUE != ''  -- очистка от пустых строк
GROUP BY SECID
ORDER BY TOTAL_VALUE DESC;
"""
result = query_sqlite('my_database.db', query=query, table_name='my_table', limit=5)

"📝 Выполняется запрос:\n\nSELECT \n    SECID,\n    SUM(CAST(VALUE AS FLOAT)) AS TOTAL_VALUE\nFROM my_table\nWHERE VALUE IS NOT NULL \n    AND VALUE != ''  -- очистка от пустых строк\nGROUP BY SECID\nORDER BY TOTAL_VALUE DESC;\n"

'📊 Результат: 547 строк'

,SECID,TOTAL_VALUE
0,RUSFARN,3.267290e+13
1,RUSFAR,1.449858e+13
2,RUPMI,3.117474e+12
3,RUPAI,3.117474e+12
4,MXTDFI2045,2.018124e+12
...,...,...
542,AKCNA,0.000000e+00
543,AKBCA,0.000000e+00
544,AKAIA,0.000000e+00
545,2xOFZ,0.000000e+00
